# Day 4 v2 — Model 05: AITeamVN/Vietnamese_Embedding + DNN (Hướng B)

**Architecture:** `AITeamVN/Vietnamese_Embedding` (frozen, 568M params) → 1024-dim dense embedding → PriceDNN head (6 ResidualBlocks)

**Tại sao AITeamVN:** VN-MTEB score cao nhất (63.34), classification score 69.06 — phù hợp price prediction theo category. BGE-M3 base, 512-token limit.

**Note:** Model nặng 568M. Dùng `encode_batch_size=64` để tránh OOM khi encode. input_size=1024 (auto-detected bởi SentTransRunner.setup).

**Target:** MAE < 70k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
import pricer_vi_2.senttrans_model as sm

# Override encoder BEFORE creating runner
sm.ENCODER_NAME = "AITeamVN/Vietnamese_Embedding"

from pricer_vi_2.senttrans_model import SentTransRunner

print(f"Encoder: {sm.ENCODER_NAME}")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Pre-compute Embeddings

AITeamVN/Vietnamese_Embedding (568M, BGE-M3 base) → 1024-dim embeddings.
`encode_batch_size=64` để tránh OOM (model nặng hơn e5-small 5x).
Lần đầu: ~15-20 phút trên GPU. Lần sau: load từ cache pkl.

> Nếu vẫn OOM: đổi `encode_batch_size=32`.

In [ ]:
runner = SentTransRunner(train, val)

cache_path = Path("cache/aitvn_embeddings.pkl")
runner.encode_and_cache(cache_path=cache_path, encode_batch_size=64)

## 3. Setup Model

DNN head: 6 ResidualBlocks. input_size=1024 (auto-detected từ embedding dim).
`batch_size=128` cho DNN training (1024-dim activations nặng hơn 384-dim).

In [ ]:
runner.setup(batch_size=128, num_blocks=6)

## 4. Train

Max 15 epochs, early stopping patience=3. Val feedback dùng val[:1000] mỗi epoch.

In [ ]:
history = runner.train(epochs=15, patience=3)

## 5. Training History

In [ ]:
plot_training_history(history, title="AITeamVN Vietnamese_Embedding + DNN")

## 6. Save Weights + Val Predictions + Test Predictions

In [ ]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/aitvn_dnn.pth")
print("Saved weights/aitvn_dnn.pth")

Path("val_predictions").mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/aitvn_val.json", "w") as f:
    json.dump(val_preds, f)

print("Encoding + predicting test set (3872 samples)...")
test_preds = runner.test_predictions(test, encode_batch_size=64)
with open("val_predictions/aitvn_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

## 7. Evaluate on 200 Test Samples

In [ ]:
def aitvn_pricer(item):
    return runner.inference(item)

results = evaluate(aitvn_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

## 8. Sanity Check — Load Roundtrip

In [ ]:
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred_original:.1f}k VND")
print(f"Error:   {abs(pred_original - sample.price):.1f}k VND")

runner.load("weights/aitvn_dnn.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch: before={pred_original:.2f} after={pred_loaded:.2f}"
print(f"\nLoad roundtrip PASSED. Diff: {diff:.4f}k")